In [1]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, TensorDataset, DataLoader
from tab_transformer_pytorch import TabTransformer, FTTransformer
from preprocessing import get_features_and_target
from sklearn.preprocessing import LabelEncoder, StandardScaler, MinMaxScaler
from RMSELoss import RMSELoss
import plotly.graph_objects as go
from tabpfn import TabPFNRegressor
from tabpfn.constants import ModelVersion
from sklearn.model_selection import train_test_split
from tabpfn_extensions import interpretability

In [2]:
import huggingface_hub
huggingface_hub.login()

# Getting Dataframe

In [3]:
# Load the training and development datasets
train_df = pd.read_csv("data/train_data.csv")
dev_df = pd.read_csv("data/development_data.csv")
#sc = StandardScaler()

target_column = "PullTest (N)"  

x_train, y_train = get_features_and_target(train_df, target_column)
x_dev, y_dev = get_features_and_target(dev_df, target_column)

#x_train = sc.fit_transform(X=x_train)
#x_dev = sc.transform(x_dev)


In [4]:
x_train

,Pressure (PSI),Welding Time (ms),Angle (Deg),Force (N),Current (A),Thickness A (mm),Thickness B (mm)
0,35,200,0,6.82,1081.47,0.922,0.920
1,35,1500,0,52.25,2014.73,0.920,0.925
2,95,200,0,16.57,1321.93,0.912,0.924
3,95,200,0,41.42,1615.83,0.948,0.939
4,35,1500,0,63.82,1137.29,0.930,0.937
...,...,...,...,...,...,...,...
290,60,1200,0,98.51,4429.61,0.625,0.622
291,60,1200,0,97.32,2636.69,0.622,0.632
292,60,1200,0,98.07,3601.67,0.666,0.633
293,60,1200,0,97.09,4161.74,0.615,0.619


# Fit Model

In [5]:
# Initialize the regressor
regressor = TabPFNRegressor()  # Uses TabPFN-2.5 weights, trained on synthetic data only.
# To use TabPFN v2:
# regressor = TabPFNRegressor.create_default_for_version(ModelVersion.V2)
regressor.fit(x_train, y_train)

# Predict on the test set
predictions = regressor.predict(x_dev)

In [6]:
predictions

array([4170.6094, 2147.6226, 4183.539 , 2147.9988, 5198.001 , 2247.6548,
       5150.5415, 2275.8022, 3338.912 , 3307.4614, 3323.2046, 2859.8506,
       2870.7544, 2861.5166, 2841.5583, 2845.353 , 2829.0354, 2845.9753,
       2734.582 , 2760.0825, 2740.755 , 2743.9438, 2734.1462, 2725.208 ,
       2732.8545, 2764.5293, 2738.727 , 2746.4873, 2744.2283, 2740.6206,
       2738.6406, 2737.606 , 2739.375 , 2741.2002, 2768.7627, 2777.7214,
       2777.1245, 2767.724 , 2754.9048, 2771.8396, 2769.1262, 2775.0776,
       2775.2925, 3106.0723, 3115.3354, 3118.0317, 3122.2124, 3121.454 ,
       3087.2866, 3074.1309, 3091.1072, 3087.0454, 3107.6287, 3053.227 ,
       3066.58  , 3060.1958, 3060.9238, 3045.0068, 3052.2034, 3054.1025,
       3059.5847, 3049.7832, 3050.669 , 2977.2   , 2974.2048, 2963.7983,
       2962.974 , 2948.1086, 2949.9773, 2961.6904, 2962.813 , 2973.7085,
       2964.2883, 2971.4338, 2965.742 , 2959.189 , 2956.5144, 2914.494 ,
       2922.9126, 2929.539 , 2926.0063, 2915.6565, 

# Check Validation Data

In [12]:
import plotly.graph_objects as go
import numpy as np

# Convert to numpy arrays
true_vals = np.array(y_dev).ravel()
pred_vals = np.array(predictions).ravel()

# Sample index
sample_idx = np.arange(len(true_vals))

# Category array (must be aligned with y_dev)
categories = dev_df.groupby("Sample ID")["Category"].first().values

# Masks for each category
mask_good    = categories == "Good"
mask_bad     = categories == "Bad"
mask_explode = categories == "Explode"

fig = go.Figure()

# --- GOOD (circles) ---
fig.add_trace(go.Scatter(
    x=sample_idx[mask_good],
    y=true_vals[mask_good],
    mode="markers",
    name="Good (True)",
    marker=dict(symbol="circle", color="red", size=7)
))

fig.add_trace(go.Scatter(
    x=sample_idx[mask_good],
    y=pred_vals[mask_good],
    mode="markers",
    name="Good (Pred)",
    marker=dict(symbol="circle", color="blue", size=7)
))

# --- BAD (X) ---
fig.add_trace(go.Scatter(
    x=sample_idx[mask_bad],
    y=true_vals[mask_bad],
    mode="markers",
    name="Bad (True)",
    marker=dict(symbol="x", color="red", size=9)
))

fig.add_trace(go.Scatter(
    x=sample_idx[mask_bad],
    y=pred_vals[mask_bad],
    mode="markers",
    name="Bad (Pred)",
    marker=dict(symbol="x", color="blue", size=9)
))

# --- EXPLODE (triangle-up) ---
fig.add_trace(go.Scatter(
    x=sample_idx[mask_explode],
    y=true_vals[mask_explode],
    mode="markers",
    name="Explode (True)",
    marker=dict(symbol="triangle-up", color="red", size=9)
))

fig.add_trace(go.Scatter(
    x=sample_idx[mask_explode],
    y=pred_vals[mask_explode],
    mode="markers",
    name="Explode (Pred)",
    marker=dict(symbol="triangle-up", color="blue", size=9)
))

# Optional: connecting lines for each sample
for i in range(len(sample_idx)):
    fig.add_trace(go.Scatter(
        x=[sample_idx[i], sample_idx[i]],
        y=[true_vals[i], pred_vals[i]],
        mode="lines",
        line=dict(color="gray", width=1),
        showlegend=False
    ))

fig.update_layout(
    title="Validation Samples: True vs Prediction (TabPFN) by Category",
    xaxis_title="Sample Index",
    yaxis_title="Pull Force",
    template="seaborn"
)

fig.show()


In [8]:
#test_x = x_dev.iloc[:50].copy()
#test_x.columns

In [9]:
#feature_names = x_train.columns.tolist()
# Fix: tell SHAP what the model's feature names are 
#regressor.feature_names_in_ = np.array(feature_names)

# Calculate SHAP values
#shap_values = interpretability.shap.get_shap_values(
#    estimator=regressor,
#    test_x=test_x,
#    attribute_names=feature_names,
#    algorithm="permutation",
#)

# Create visualization
#fig = interpretability.shap.plot_shap(shap_values)

In [10]:
#x_dev.columns


# Check Validation Loss and R2

In [11]:
# Calculate MAE and RMSE and R2
mae  = mean_absolute_error(y_dev, predictions)
rmse = np.sqrt(mean_squared_error(y_dev, predictions))
R2   = r2_score(y_dev, predictions)


print(f"MAE:  {mae:.2f}")
print(f"RMSE: {rmse:.2f}")
print(f"R2: {R2:.2f}")


MAE:  126.26
RMSE: 219.93
R2: 0.62
